# 06. Creating Qdrant Database

Upload chunked embeddings into Qdrant vector database with optimized configuration:
- HNSW index tuning for N vectors
- Payload indexes for filtered search
- Streaming upload via PyArrow (no full DataFrame in RAM)
- Snapshot creation for deployment

In [1]:
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, 
    VectorParams,
    HnswConfigDiff,
    OptimizersConfigDiff,
    PayloadSchemaType,
    PointStruct,
)
from tqdm.auto import tqdm

In [2]:
# === CONFIG ===
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

COLLECTION_NAME = "nlp2025_chunks"
VECTOR_SIZE = 1024  # Qwen3-Embedding-0.6B output dim

PARQUET_PATH = Path("../../data/processed/all_chunks_with_embeddings.parquet")
SNAPSHOT_DIR = Path("../../")

BATCH_SIZE = 1024  # Points per upsert call
ARROW_BATCH_SIZE = 4096  # Rows per PyArrow read batch

In [3]:
# 1. Connect to Qdrant & verify
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT, timeout=300)

cluster_info = client.get_collections()
print(f"Connected to Qdrant. Existing collections: {[c.name for c in cluster_info.collections]}")

Connected to Qdrant. Existing collections: ['nlp2025_chunks']


## 1. Create Collection

Configuration rationale:
- **HNSW `m=16, ef_construct=128`** — balanced recall/speed for ~658K vectors
- **`on_disk=True` for vectors** — keeps vectors on SSD, frees RAM for HNSW graph
- **`on_disk_payload=True`** — payload (text, metadata) served from disk
- **`indexing_threshold=50_000`** — delays HNSW rebuilds until bulk upload completes

In [4]:
# NOTE: Drop if exists (clean slate)
if client.collection_exists(collection_name=COLLECTION_NAME):
    client.delete_collection(collection_name=COLLECTION_NAME)
    print(f"Collection '{COLLECTION_NAME}' deleted.")

# Create with tuned parameters
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_SIZE,
        distance=Distance.COSINE,
        on_disk=True,  # Vectors on NVMe, not in RAM
    ),
    hnsw_config=HnswConfigDiff(
        m=16,
        ef_construct=128,
        on_disk=True,  # HNSW graph on disk too
    ),
    optimizers_config=OptimizersConfigDiff(
        indexing_threshold=50_000,  # Delay index build during bulk upload
    ),
    on_disk_payload=True,  # Payload on disk
)

print(f"Collection '{COLLECTION_NAME}' created.")

Collection 'nlp2025_chunks' deleted.
Collection 'nlp2025_chunks' created.


## 2. Create Payload Indexes

Indexes enable fast filtered search:
- `doc_id` — filter by specific paper
- `Header_1` — filter by paper title (top-level header)

In [5]:
# Payload indexes for filtered search
for field_name, field_type in [
    ("doc_id", PayloadSchemaType.KEYWORD),
    ("Header_1", PayloadSchemaType.KEYWORD),
]:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field_name,
        field_schema=field_type,
    )
    print(f"Index created: {field_name} ({field_type})")

Index created: doc_id (keyword)
Index created: Header_1 (keyword)


## 3. Upload Data

Stream data via PyArrow `read_table` + batch iteration.
This avoids loading the entire parquet into a pandas DataFrame.

In [6]:
# Quick inspection: schema & row count
parquet_meta = pq.read_metadata(PARQUET_PATH)
table_schema = pq.read_schema(PARQUET_PATH)

total_rows = parquet_meta.num_rows
print(f"Parquet: {total_rows:,} rows")
print(f"Schema: {table_schema}")

Parquet: 33,307 rows
Schema: doc_id: string
text: string
embed_text: string
metadata: struct<authors: string, chunk_index: int64, header_1: string, header_2: string, header_3: string, pu (... 31 chars omitted)
  child 0, authors: string
  child 1, chunk_index: int64
  child 2, header_1: string
  child 3, header_2: string
  child 4, header_3: string
  child 5, published: string
  child 6, title: string
embedding: list<element: float>
  child 0, element: float
-- schema metadata --
huggingface: '{"info": {"features": {"doc_id": {"dtype": "string", "_type' + 567


In [7]:
def iter_points(parquet_path: Path, arrow_batch_size: int = 4096):
    """
    Generator: yields PointStruct one-by-one from a parquet file.
    Reads data in Arrow batches to minimize RAM usage.
    """
    table = pq.read_table(parquet_path)
    point_id = 0

    for batch in table.to_batches(max_chunksize=arrow_batch_size):
        # Convert batch columns to Python-friendly formats
        texts = batch.column("text").to_pylist()
        doc_ids = batch.column("doc_id").to_pylist()
        embeddings = batch.column("embedding").to_pylist()
        metadata_col = batch.column("metadata").to_pylist()

        for text, doc_id, embedding, meta in zip(texts, doc_ids, embeddings, metadata_col):
            # Build payload: text + doc_id + flattened metadata
            payload = {
                "text": text,
                "doc_id": doc_id,
            }

            # Safely unpack metadata dict (could be None/NaN)
            if isinstance(meta, dict):
                payload.update(meta)

            # Ensure float32 list for Qdrant
            vector = np.array(embedding, dtype=np.float32).tolist()

            yield PointStruct(
                id=point_id,
                vector=vector,
                payload=payload,
            )
            point_id += 1

In [8]:
# Batch upload with progress bar
batch = []
uploaded = 0

pbar = tqdm(total=total_rows, desc="Uploading", unit="pts")

for point in iter_points(PARQUET_PATH, arrow_batch_size=ARROW_BATCH_SIZE):
    batch.append(point)

    if len(batch) >= BATCH_SIZE:
        client.upsert(
            collection_name=COLLECTION_NAME,
            points=batch,
            wait=True,
        )
        uploaded += len(batch)
        pbar.update(len(batch))
        batch = []

# Flush remaining
if batch:
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True,
    )
    uploaded += len(batch)
    pbar.update(len(batch))

pbar.close()
print(f"Upload complete: {uploaded:,} points")

Uploading:   0%|          | 0/33307 [00:00<?, ?pts/s]

Upload complete: 33,307 points


## 4. Restore Indexing & Optimize

Lower `indexing_threshold` back to default so Qdrant starts building HNSW segments.

In [9]:
# Restore default indexing threshold to trigger HNSW build
client.update_collection(
    collection_name=COLLECTION_NAME,
    optimizer_config=OptimizersConfigDiff(
        indexing_threshold=20_000,  # Qdrant default
    ),
)
print("Indexing threshold restored. HNSW index will build in background.")

Indexing threshold restored. HNSW index will build in background.


## 5. Validation

In [10]:
# Verify point count matches source
info = client.get_collection(collection_name=COLLECTION_NAME)

print(f"Collection: {COLLECTION_NAME}")
print(f"  Points:          {info.points_count:,}")
print(f"  Indexed vectors: {info.indexed_vectors_count:,}")
print(f"  Status:          {info.status}")
print(f"  Segments:        {info.segments_count}")

assert info.points_count == total_rows, (
    f"MISMATCH: expected {total_rows:,}, got {info.points_count:,}"
)
print(f"\n[OK] Count matches source parquet ({total_rows:,} rows).")

Collection: nlp2025_chunks
  Points:          33,307
  Indexed vectors: 0
  Status:          yellow
  Segments:        6

[OK] Count matches source parquet (33,307 rows).


In [11]:
# Inspect a random point's payload structure
import random

sample_ids = random.sample(range(total_rows), k=3)
sample_points = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=sample_ids,
    with_payload=True,
    with_vectors=False,
)

print("Sample payload structure:")
for pt in sample_points:
    print(f"\n  ID: {pt.id}")
    print(f"  Payload keys: {list(pt.payload.keys())}")
    text_preview = pt.payload.get('text', '')[:120]
    print(f"  Text preview: {text_preview}...")
    print(f"  doc_id: {pt.payload.get('doc_id', 'N/A')}")
    print(f"  Header_1: {pt.payload.get('Header_1', 'N/A')}")

Sample payload structure:

  ID: 7766
  Payload keys: ['text', 'doc_id', 'authors', 'chunk_index', 'header_1', 'header_2', 'header_3', 'published', 'title']
  Text preview: This defines $(ab)^{*}$ .  
But a formula in ${\mathsf{TL}}[\mathord{{\mathbf{Y}}}]$ cannot distinguish between strings ...
  doc_id: 2510.27118v4
  Header_1: N/A

  ID: 24226
  Payload keys: ['text', 'doc_id', 'authors', 'chunk_index', 'header_1', 'header_2', 'header_3', 'published', 'title']
  Text preview: Drawing from existing research(Yang and Jin, 2024; Chakrabarty et al., 2024a), we evaluate story quality across seven na...
  doc_id: 2509.26461v1
  Header_1: N/A

  ID: 12677
  Payload keys: ['text', 'doc_id', 'authors', 'chunk_index', 'header_1', 'header_2', 'header_3', 'published', 'title']
  Text preview: 20Option D: Children have political subjecthood that is missed when they are considered as passive victims of warfare.  ...
  doc_id: 2502.00089v1
  Header_1: N/A


## 6. Test Search

In [12]:
# Sanity check: search with a vector from the dataset
# Read just one row to get a test vector
table_sample = pq.read_table(PARQUET_PATH, columns=["embedding", "text"])
test_embedding = np.array(table_sample.column("embedding")[0].as_py(), dtype=np.float32).tolist()
test_text = table_sample.column("text")[0].as_py()

print(f"Query text: {test_text[:100]}...\n")

hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=test_embedding,
    limit=5,
    with_payload=["text", "doc_id", "Header_1"],  # Request only needed fields
)

print("Top-5 results:")
for rank, hit in enumerate(hits.points, 1):
    text_preview = hit.payload.get('text', '')[:100]
    print(f"  #{rank} | score={hit.score:.4f} | id={hit.id}")
    print(f"       doc_id={hit.payload.get('doc_id', 'N/A')}")
    print(f"       text: {text_preview}...")
    print()

Query text: ###### Abstract  
Despite the growing reasoning capabilities of recent large language models (LLMs),...

Top-5 results:
  #1 | score=1.0000 | id=0
       doc_id=2512.23988v1
       text: ###### Abstract  
Despite the growing reasoning capabilities of recent large language models (LLMs),...

  #2 | score=0.8929 | id=3
       doc_id=2512.23988v1
       text: Unlike prior approaches, our framework does not rely on human-supervised labels; instead, it directl...

  #3 | score=0.8897 | id=20
       doc_id=2512.23988v1
       text: We presented an unsupervised framework for uncovering and manipulating reasoning behaviors in LLMs. ...

  #4 | score=0.8572 | id=2
       doc_id=2512.23988v1
       text: $v=\frac{1}{N_{+}}\sum_{i=1}^{N_{+}}h^{+}_{i}\;-\;\frac{1}{N_{-}}\sum_{j=1}^{N_{-}}h^{-}_{j}$ . The ...

  #5 | score=0.7480 | id=4
       doc_id=2512.23988v1
       text: Reasoning Models. Previous works have demonstrated that enabling models to generate longer outputs c...

